# Day 25: Logistic Regression Classification

**Dataset:** `Bank_Marketing_Dataset.csv`

This notebook builds a logistic regression model for binary classification and evaluates its
predictions on unseen data, predicting
whether a customer subscribed to a term `deposit` (`yes`/`no`).

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the dataset
df = pd.read_csv("../datasets/Bank_Marketing_Dataset.csv")

# Preview the data
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
print(df.shape)
df.info()

(11162, 17)
<class 'pandas.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        11162 non-null  int64
 1   job        11162 non-null  str  
 2   marital    11162 non-null  str  
 3   education  11162 non-null  str  
 4   default    11162 non-null  str  
 5   balance    11162 non-null  int64
 6   housing    11162 non-null  str  
 7   loan       11162 non-null  str  
 8   contact    11162 non-null  str  
 9   day        11162 non-null  int64
 10  month      11162 non-null  str  
 11  duration   11162 non-null  int64
 12  campaign   11162 non-null  int64
 13  pdays      11162 non-null  int64
 14  previous   11162 non-null  int64
 15  poutcome   11162 non-null  str  
 16  deposit    11162 non-null  str  
dtypes: int64(7), str(10)
memory usage: 1.4 MB


In [3]:
print(df.isna().sum().sum(), "missing values")
print()
print("Target balance:")
print(df["deposit"].value_counts(normalize=True).round(3))

0 missing values

Target balance:
deposit
no     0.526
yes    0.474
Name: proportion, dtype: float64


## 2. Preprocessing the Dataset

**Encode the target: `deposit` (yes/no) → 1/0**

In [4]:
df["deposit"] = df["deposit"].map({"yes": 1, "no": 0})
df["deposit"].value_counts()

deposit
0    5873
1    5289
Name: count, dtype: int64

**Encode simple yes/no predictor columns the same way**

In [5]:
binary_columns = ["default", "housing", "loan"]

for column in binary_columns:
    df[column] = df[column].map({"yes": 1, "no": 0})

df[binary_columns].head()

,default,housing,loan
0,0,1,0
1,0,0,0
2,0,1,0
3,0,1,0
4,0,0,0


**One-hot encode the remaining multi-category text columns**

In [6]:
categorical_columns = ["job", "marital", "education", "contact", "month", "poutcome"]

df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print("Shape after encoding:", df.shape)
df.head()

Shape after encoding: (11162, 43)


,age,default,balance,housing,loan,day,duration,campaign,pdays,previous,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,59,0,2343,1,0,5,1042,1,-1,0,...,False,False,False,True,False,False,False,False,False,True
1,56,0,45,0,0,5,1467,1,-1,0,...,False,False,False,True,False,False,False,False,False,True
2,41,0,1270,1,0,5,1389,1,-1,0,...,False,False,False,True,False,False,False,False,False,True
3,55,0,2476,1,0,5,579,1,-1,0,...,False,False,False,True,False,False,False,False,False,True
4,54,0,184,0,0,5,673,2,-1,0,...,False,False,False,True,False,False,False,False,False,True


**Separate predictors and target**

In [7]:
X = df.drop(columns=["deposit"])
y = df["deposit"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11162, 42)
y shape: (11162,)


**Train-test split (stratified, to keep the same yes/no balance in both sets)**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Training set size: 8929
Test set size: 2233


**Scale the numerical features**

Logistic regression's optimization (and the size of its coefficients) is sensitive to
feature scale, so the numeric columns are standardized, fitting the scaler only on the
training data to avoid leaking test-set information.

In [9]:
numeric_columns = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_test_scaled[numeric_columns] = scaler.transform(X_test[numeric_columns])

X_train_scaled[numeric_columns].describe().round(2)

,age,balance,day,duration,campaign,pdays,previous
count,8929.00,8929.00,8929.00,8929.00,8929.00,8929.00,8929.00
mean,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.95,-2.80,-1.74,-1.06,-0.54,-0.48,-0.36
25%,-0.77,-0.46,-0.91,-0.67,-0.54,-0.48,-0.36
50%,-0.18,-0.31,-0.08,-0.34,-0.18,-0.48,-0.36
75%,0.66,0.07,0.75,0.35,0.18,-0.13,0.07
max,4.54,26.80,1.82,8.42,21.73,7.35,24.52


## 3. Building the Logistic Regression Model

In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("Model trained.")

Model trained.


## 4. Predictions

In [11]:
y_pred = model.predict(X_test_scaled)

predictions_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred
}).reset_index(drop=True)

predictions_df.head(10)

,actual,predicted
0,0,1
1,1,1
2,0,1
3,0,0
4,1,1
5,0,0
6,0,0
7,1,0
8,1,1
9,1,1


**`predict_proba()` — inspecting the underlying predicted probabilities**

`predict()` just applies a 0.5 threshold to these probabilities to get a final class.

In [12]:
probabilities = model.predict_proba(X_test_scaled)

proba_df = pd.DataFrame({
    "prob_no (class 0)": probabilities[:, 0],
    "prob_yes (class 1)": probabilities[:, 1],
    "predicted_class": y_pred
}).round(3)

proba_df.head(10)

,prob_no (class 0),prob_yes (class 1),predicted_class
0,0.441,0.559,1
1,0.320,0.680,1
2,0.345,0.655,1
3,0.938,0.062,0
4,0.235,0.765,1
5,0.987,0.013,0
6,0.888,0.112,0
7,0.621,0.379,0
8,0.005,0.995,1
9,0.109,0.891,1


## 5. Accuracy and Classification Report

In [13]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

Accuracy: 0.825

Classification Report:
              precision    recall  f1-score   support

          no       0.82      0.85      0.84      1175
         yes       0.83      0.80      0.81      1058

    accuracy                           0.83      2233
   macro avg       0.83      0.82      0.82      2233
weighted avg       0.83      0.83      0.83      2233



In [14]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("[[True Negatives, False Positives]")
print(" [False Negatives, True Positives]]")

Confusion Matrix:
[[999 176]
 [214 844]]
[[True Negatives, False Positives]
 [False Negatives, True Positives]]


## 6. Short Model Interpretation

In [15]:
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

coefficients.head(10)

,feature,coefficient
40,poutcome_success,2.181904
6,duration,1.858876
34,month_mar,1.756042
27,contact_unknown,-1.500596
29,month_dec,1.209796
31,month_jan,-1.168689
32,month_jul,-1.048689
36,month_nov,-1.002695
28,month_aug,-0.873429
37,month_oct,0.856910


**Interpretation:**

- **`poutcome_success`** has the single largest positive coefficient, i.e., customers whose
  *previous* marketing campaign outcome was a success are far more likely to subscribe again,
  which makes strong intuitive sense.
- **`duration`** (call duration) is also one of the strongest predictors, but this comes with
  an important caveat: a call's duration is only known *after* the call has already happened,
  so it's not something you'd actually have available if predicting *before* calling a
  customer. In a real deployment, `duration` would typically be dropped to avoid this kind of
  data leakage, even though it strongly improves this model's apparent accuracy.
- Several **`month_*`** dummy variables have large coefficients in both directions, showing a
  genuine seasonal pattern in subscription likelihood. That is, some months (like March and December)
  are associated with a higher chance of subscribing, while others (like January and July) are
  associated with a lower chance.
- The model reached **~82.5% accuracy** with balanced precision and recall for both classes,
  on a dataset where the two classes are themselves fairly balanced (≈53% no / 47% yes). So,
  this accuracy reflects real predictive signal, not just the model favoring the majority
  class.

## Outcome

This notebook built a logistic regression model to predict whether a customer subscribed to a
term deposit, using the `Bank_Marketing_Dataset.csv` file. Preprocessing involved encoding the
binary target and binary predictor columns to 0/1, one-hot encoding six multi-category text
columns, and standardizing the numeric features using a scaler fit only on the training data.
The trained model reached about 82.5% accuracy on the held-out test set, with balanced
precision and recall for both classes. Using `predict_proba()` showed the underlying
probability behind each prediction rather than just the final yes/no label, and examining the
model's coefficients revealed that a successful previous campaign outcome and seasonal timing
(month) were strong real predictors while also surfacing an important real-world caveat
about `duration` representing data that wouldn't actually be available at prediction time in a
genuine deployment scenario.